In [4]:

"""
select_most_unique_images.py

Finds the most "unique" images in one or more input folders using
OpenCLIP embeddings + ChromaDB.

For each image:
    - Compare its embedding against every other image.
    - Calculate the average distance to all other images.
    - Images with HIGHER average distance are considered more unique.
    - Copy the top N most unique images to the output folder.

Requirements:
    pip install chromadb open-clip-torch torch pillow tqdm pandas

Usage:
    python select_most_unique_images.py
"""

from pathlib import Path
import shutil

import chromadb
import open_clip
import torch
from PIL import Image
from tqdm import tqdm
import pandas as pd
import os


#from pathlib import Path 
#TRAIN_DIR = Path("/Users/boy/Desktop/licenseplate-dataset/train")
#OUTPUT_FOLDER = Path("./OUTPUT")
#os.makedirs(OUTPUT_FOLDER, exist_ok = True)

# ============================================================
# CONFIGURATION
# ============================================================

# You can provide multiple input folders
INPUT_FOLDERS = [
    r"/Users/boy/Desktop/licenseplate-dataset/train"
]

OUTPUT_FOLDER = r"./selected_images"
os.makedirs(OUTPUT_FOLDER, exist_ok = True)
# Number of most unique images to select
TOP_N = 10

# ChromaDB temporary/persistent database
CHROMA_DB_PATH = r"./chroma_db"

# Chroma collection name
COLLECTION_NAME = "image_embeddings"

# CLIP model
MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Supported image extensions
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
    ".tif",
    ".tiff",
}


# ============================================================
# FIND IMAGES
# ============================================================

def find_images(input_folders):
    """Find all images recursively from input folders."""

    image_paths = []

    for folder in input_folders:
        folder = Path(folder)

        if not folder.exists():
            print(f"WARNING: Folder does not exist: {folder}")
            continue

        for path in folder.rglob("*"):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                image_paths.append(path)

    # Remove duplicates while preserving order
    image_paths = list(dict.fromkeys(image_paths))

    return image_paths


# ============================================================
# LOAD CLIP
# ============================================================

def load_model():
    print(f"Using device: {DEVICE}")
    print(f"Loading CLIP model: {MODEL_NAME}")

    model, _, preprocess = open_clip.create_model_and_transforms(
        MODEL_NAME,
        pretrained=PRETRAINED,
        device=DEVICE,
    )

    model.eval()

    return model, preprocess


# ============================================================
# GENERATE EMBEDDINGS
# ============================================================

def generate_embeddings(image_paths, model, preprocess):
    """Generate normalized CLIP embeddings."""

    embeddings = []
    valid_paths = []

    print("\nGenerating image embeddings...")

    with torch.no_grad():

        for image_path in tqdm(image_paths):

            try:
                image = Image.open(image_path).convert("RGB")
                image_tensor = preprocess(image).unsqueeze(0).to(DEVICE)

                embedding = model.encode_image(image_tensor)

                # Normalize embedding
                embedding = embedding / embedding.norm(
                    dim=-1,
                    keepdim=True
                )

                embedding = embedding.cpu().numpy()[0].tolist()

                embeddings.append(embedding)
                valid_paths.append(image_path)

            except Exception as e:
                print(f"\nSkipping {image_path}")
                print(f"Reason: {e}")

    return valid_paths, embeddings


# ============================================================
# CREATE CHROMA DATABASE
# ============================================================

def create_chroma_collection():
    client = chromadb.PersistentClient(
        path=CHROMA_DB_PATH
    )

    # Delete old collection if it exists
    try:
        client.delete_collection(COLLECTION_NAME)


    except Exception:
        
        pass
    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={
            "hnsw:space": "cosine"
        }
    )

    return collection


# ============================================================
# ADD EMBEDDINGS TO CHROMA
# ============================================================

def add_embeddings_to_chroma(
    collection,
    image_paths,
    embeddings
):
    print("\nAdding embeddings to ChromaDB...")

    ids = [
        str(i)
        for i in range(len(image_paths))
    ]

    documents = [
        str(path)
        for path in image_paths
    ]

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=documents,
    )


# ============================================================
# CALCULATE AVERAGE DISTANCES
# ============================================================

def calculate_average_distances(
    collection,
    image_paths,
):
    """
    For each image:
        Query all other images.
        Calculate average distance.

    Higher average distance = more unique image.
    """

    results = []

    total_images = len(image_paths)

    print("\nCalculating distances...")

    for index in tqdm(range(total_images)):

        # Query all images
        query_result = collection.query(
            query_embeddings=[
                collection.get(
                    ids=[str(index)],
                    include=["embeddings"]
                )["embeddings"][0]
            ],
            n_results=total_images,
            include=[
                "distances",
                "documents",
            ],
        )

        distances = query_result["distances"][0]
        documents = query_result["documents"][0]

        # Remove the image itself.
        # Its distance to itself should be ~0.
        other_distances = []

        for distance, document in zip(
            distances,
            documents
        ):
            if document != str(image_paths[index]):
                other_distances.append(distance)

        if other_distances:
            average_distance = sum(other_distances) / len(
                other_distances
            )
        else:
            average_distance = 0.0

        results.append({
            "image": str(image_paths[index]),
            "average_distance": average_distance,
            "num_comparisons": len(other_distances),
        })

    return results


# ============================================================
# SAVE TOP IMAGES
# ============================================================

def save_top_images(results, output_folder, top_n):
    output_folder = Path(output_folder)

    output_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    # Sort highest average distance first
    results = sorted(
        results,
        key=lambda x: x["average_distance"],
        reverse=True,
    )

    selected = results[:top_n]

    print("\n" + "=" * 70)
    print(f"TOP {len(selected)} MOST UNIQUE IMAGES")
    print("=" * 70)

    for rank, item in enumerate(selected, start=1):

        source = Path(item["image"])

        # Add rank to filename
        destination = output_folder / (
            f"{rank:02d}_"
            f"{source.name}"
        )

        shutil.copy2(
            source,
            destination
        )

        print(
            f"{rank:02d}. "
            f"{source.name} "
            f"(average distance: "
            f"{item['average_distance']:.6f})"
        )

    return selected


# ============================================================
# SAVE CSV
# ============================================================

def save_results_csv(results):
    df = pd.DataFrame(results)

    df = df.sort_values(
        "average_distance",
        ascending=False
    )

    csv_path = Path(OUTPUT_FOLDER) / "image_ranking.csv"

    df.to_csv(
        csv_path,
        index=False
    )

    print(f"\nRanking saved to: {csv_path}")


# ============================================================
# MAIN


# ============================================================

def main():

    print("=" * 70)
    print("IMAGE UNIQUENESS SELECTOR")
    print("=" * 70)

    # --------------------------------------------------------
    # Find images
    # --------------------------------------------------------

    image_paths = find_images(INPUT_FOLDERS)

    print(f"\nFound {len(image_paths)} images.")

    if len(image_paths) < 2:
        print("Need at least 2 images.")
        return

    # --------------------------------------------------------
    # Load CLIP
    # --------------------------------------------------------

    model, preprocess = load_model()

    # --------------------------------------------------------
    # Generate embeddings
    # --------------------------------------------------------

    image_paths, embeddings = generate_embeddings(
        image_paths,
        model,
        preprocess,
    )

    if len(image_paths) < 2:
        print("Not enough valid images.")
        return

    # --------------------------------------------------------
    # Create ChromaDB
    # --------------------------------------------------------

    collection = create_chroma_collection()

    # --------------------------------------------------------
    # Store embeddings
    # --------------------------------------------------------

    add_embeddings_to_chroma(
        collection,
        image_paths,
        embeddings,
    )

    # --------------------------------------------------------
    # Calculate average distances
    # --------------------------------------------------------

    results = calculate_average_distances(
        collection,
        image_paths,
    )

    # --------------------------------------------------------
    # Save top images
    # --------------------------------------------------------

    selected = save_top_images(
        results,
        OUTPUT_FOLDER,
        TOP_N,
    )

    # --------------------------------------------------------
    # Save ranking
    # --------------------------------------------------------

    save_results_csv(results)

    print("\nDone.")
    print(f"Selected images: {len(selected)}")
    print(f"Output folder: {OUTPUT_FOLDER}")



In [5]:
main()

IMAGE UNIQUENESS SELECTOR

Found 98798 images.
Using device: cpu
Loading CLIP model: ViT-B-32

Generating image embeddings...


100%|█████████████████████| 98798/98798 [1:06:08<00:00, 24.90it/s]



Adding embeddings to ChromaDB...


InternalError: ValueError: Batch size of 98798 is greater than max batch size of 5461